# LegacyAgent V1
# Day 4 — Training Pipeline

This is where I get the dataset ready for actually fine-tuning Qwen3-ASR.

Pipeline so far:
Dataset → Baseline Evaluation → Data Audit → Training Pipeline → QLoRA Training

What I'm doing in this notebook:

- Load the audited manifests
- Drop samples that need alignment
- Build Hugging Face datasets from them
- Check that audio loads correctly
- Prep the processor inputs
- Run through the training pipeline end-to-end to make sure it works

In [2]:
!pip -q install -U datasets transformers accelerate torchaudio librosa soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.0 MB/s eta 0:00:00


In [3]:
from google.colab import drive

from pathlib import Path
import json
import random
import numpy as np

import torch
import torchaudio

from datasets import Dataset

In [4]:
drive.mount("/content/drive")

Mounted at /content/drive


In [5]:
ROOT = Path("/content/drive/MyDrive/legacyagent")

MANIFEST_DIR = ROOT / "manifests"

TRAIN_MANIFEST = MANIFEST_DIR / "train_manifest.jsonl"
VAL_MANIFEST = MANIFEST_DIR / "validation_manifest.jsonl"

assert ROOT.exists()
assert TRAIN_MANIFEST.exists()
assert VAL_MANIFEST.exists()

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("=" * 70)
print("LEGACYAGENT — DAY 4 TRAINING PIPELINE")
print("=" * 70)
print("Project Root :", ROOT)
print("Train Manifest :", TRAIN_MANIFEST.name)
print("Validation Manifest :", VAL_MANIFEST.name)

LEGACYAGENT — DAY 4 TRAINING PIPELINE
Project Root : /content/drive/MyDrive/legacyagent
Train Manifest : train_manifest.jsonl
Validation Manifest : validation_manifest.jsonl


In [6]:
def load_manifest(path):

    rows = []

    with open(path, "r") as f:

        for line in f:

            line = line.strip()

            if line:
                rows.append(json.loads(line))

    return rows

In [7]:
train_rows = load_manifest(TRAIN_MANIFEST)
val_rows = load_manifest(VAL_MANIFEST)

print("Train Samples :", len(train_rows))
print("Validation Samples :", len(val_rows))

Train Samples : 3702
Validation Samples : 903


In [8]:
train_ready = [
    row for row in train_rows
    if not row["needs_alignment"]
]

val_ready = [
    row for row in val_rows
    if not row["needs_alignment"]
]

print("="*60)

print("Training Ready Statistics")

print("="*60)

print("Original Train      :", len(train_rows))
print("Training Ready      :", len(train_ready))

print()

print("Original Validation :", len(val_rows))
print("Validation Ready    :", len(val_ready))


Training Ready Statistics
Original Train      : 3702
Training Ready      : 3638

Original Validation : 903
Validation Ready    : 885


In [9]:
assert all(
    not row["needs_alignment"]
    for row in train_ready
)

assert all(
    not row["needs_alignment"]
    for row in val_ready
)

print("PASS")
print("All remaining samples are training-ready.")

PASS
All remaining samples are training-ready.


In [10]:
sample = train_ready[0]

print("="*70)

for key, value in sample.items():

    print(f"{key:18}: {value}")

segment_id        : 2011_10-704_000000
case_id           : 2011_10-704
case_name         : Messerschmidt v. Millender
audio_path        : /content/drive/MyDrive/legacyagent/raw_audio/2011_10-704.wav
start             : 0.0
end               : 9.022
duration          : 9.022
text              : We will hear argument next in Case 10-704, Messerschmidt v. Millender. Mr. Coates.
speaker_id        : john_g_roberts_jr
speaker_name      : John G. Roberts, Jr.
speaker_role      : scotus_justice
section_index     : 0
turn_index        : 0
split             : train
needs_alignment   : False


# Build Hugging Face Dataset

Turning the audited JSONL manifests into Hugging Face datasets, keeping all the metadata I'll need later for training and eval.

In [11]:
train_dataset = Dataset.from_list(train_ready)
val_dataset = Dataset.from_list(val_ready)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['segment_id', 'case_id', 'case_name', 'audio_path', 'start', 'end', 'duration', 'text', 'speaker_id', 'speaker_name', 'speaker_role', 'section_index', 'turn_index', 'split', 'needs_alignment'],
    num_rows: 3638
})
Dataset({
    features: ['segment_id', 'case_id', 'case_name', 'audio_path', 'start', 'end', 'duration', 'text', 'speaker_id', 'speaker_name', 'speaker_role', 'section_index', 'turn_index', 'split', 'needs_alignment'],
    num_rows: 885
})


# Audio Loading

Each manifest row has an audio_path plus a start/end time. Instead of loading the whole courtroom recording, I just pull the specific segment I need.

In [12]:
TARGET_SR = 16000

resampler = None


def load_audio_segment(example):

    global resampler

    waveform, sr = torchaudio.load(example["audio_path"])

    start = int(example["start"] * sr)
    end = int(example["end"] * sr)

    waveform = waveform[:, start:end]

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    if sr != TARGET_SR:

        if resampler is None or resampler.orig_freq != sr:

            resampler = torchaudio.transforms.Resample(
                sr,
                TARGET_SR
            )

        waveform = resampler(waveform)

    waveform = waveform.squeeze(0)

    return {
        "audio": waveform.numpy(),
        "sampling_rate": TARGET_SR
    }

In [13]:
sample = train_dataset[0]

audio = load_audio_segment(sample)

print("Audio Length :", len(audio["audio"]))
print("Sampling Rate:", audio["sampling_rate"])

print(
    "Duration:",
    round(
        len(audio["audio"]) /
        audio["sampling_rate"],
        2
    ),
    "seconds"
)

Audio Length : 144352
Sampling Rate: 16000
Duration: 9.02 seconds


In [14]:
expected = sample["duration"]

actual = len(audio["audio"]) / audio["sampling_rate"]

print("Manifest :", round(expected, 3))
print("Loaded   :", round(actual, 3))

assert abs(expected - actual) < 0.05

print("PASS")

Manifest : 9.022
Loaded   : 9.022
PASS


In [15]:
from IPython.display import Audio

Audio(
    audio["audio"],
    rate=audio["sampling_rate"]
)

# Load Qwen3-ASR Processor

Loading the tokenizer and feature extractor that turn audio waveforms and transcripts into model-ready inputs. Same processor gets reused for both preprocessing and training.

In [16]:
from transformers import AutoProcessor

MODEL_NAME = "Qwen/Qwen3-ASR-1.7B"

processor = AutoProcessor.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True
)

print(type(processor))
print("\nProcessor loaded successfully.")

preprocessor_config.json:   0%|          | 0.00/330 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/6.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/12.5k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

<class 'transformers.models.qwen3_asr.processing_qwen3_asr.Qwen3ASRProcessor'>

Processor loaded successfully.


In [17]:
print(processor)

print("\nTokenizer:")
print(type(processor.tokenizer))

print("\nFeature Extractor:")
print(type(processor.feature_extractor))

Qwen3ASRProcessor:
- feature_extractor: WhisperFeatureExtractor {
  "chunk_length": 30,
  "dither": 0.0,
  "feature_extractor_type": "WhisperFeatureExtractor",
  "feature_size": 128,
  "hop_length": 160,
  "n_fft": 400,
  "n_samples": 480000,
  "nb_max_frames": 3000,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": true,
  "sampling_rate": 16000
}

- tokenizer: Qwen2Tokenizer(name_or_path='Qwen/Qwen3-ASR-1.7B', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>', 'audio_bos_token': '<|audio_start|>', 'audio_eos_token': '<|audio_end|>', 'audio_token': '<|audio_pad|>', 'image_token': '<|image_pad|>', 'video_token': '<|video_pad|>', 'vision_bos_token': '<|vision_start|>', 'vision_eos_token': '<|vision_end|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	15

# Feature Extraction Test

Before running this on the whole dataset, let me check the processor converts one audio sample into acoustic features correctly -- just confirming the audio pipeline actually works with Qwen3-ASR.

In [18]:
sample = train_dataset[0]

audio = load_audio_segment(sample)

features = processor.feature_extractor(
    audio["audio"],
    sampling_rate=audio["sampling_rate"],
    return_tensors="pt"
)

print("Feature keys:")
print(features.keys())

print("\nInput feature shape:")
print(features.input_features.shape)

Feature keys:
KeysView({'input_features': tensor([[[-0.5327, -0.5327, -0.5327,  ..., -0.5327, -0.5327, -0.5327],
         [-0.5327, -0.5327, -0.5327,  ..., -0.5327, -0.5327, -0.5327],
         [-0.5327, -0.5327, -0.5327,  ..., -0.5327, -0.5327, -0.5327],
         ...,
         [-0.5327, -0.5327, -0.5327,  ..., -0.5327, -0.5327, -0.5327],
         [-0.5327, -0.5327, -0.5327,  ..., -0.5327, -0.5327, -0.5327],
         [-0.5327, -0.5327, -0.5327,  ..., -0.5327, -0.5327, -0.5327]]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0]], dtype=torch.int32)})

Input feature shape:
torch.Size([1, 128, 3000])


In [19]:
transcript = sample["text"]

labels = processor.tokenizer(
    transcript,
    return_tensors="pt"
)

print("Transcript:")
print(transcript)

print("\nToken shape:")
print(labels.input_ids.shape)

print("\nFirst 20 token ids:")
print(labels.input_ids[0][:20])

Transcript:
We will hear argument next in Case 10-704, Messerschmidt v. Millender. Mr. Coates.

Token shape:
torch.Size([1, 29])

First 20 token ids:
tensor([ 1654,   686,  6723,  5693,  1790,   304, 11538,   220,    16,    15,
           12,    22,    15,    19,    11, 18713,   388,   331, 42301,   348])


# Lightweight Dataset

Rather than precomputing acoustic features for every example and holding them all in memory, I'm just keeping the metadata needed for training. Audio features get generated on the fly during batching instead -- keeps RAM usage way down and scales better to larger datasets.

In [20]:
train_dataset = train_dataset.remove_columns(
    [c for c in train_dataset.column_names if c not in ["audio_path", "start", "end", "text"]]
)

val_dataset = val_dataset.remove_columns(
    [c for c in val_dataset.column_names if c not in ["audio_path", "start", "end", "text"]]
)

print(train_dataset)
print(val_dataset)

Dataset({
    features: ['audio_path', 'start', 'end', 'text'],
    num_rows: 3638
})
Dataset({
    features: ['audio_path', 'start', 'end', 'text'],
    num_rows: 885
})


# Notebook Summary

Dataset's validated and converted into a lightweight format that just holds the metadata needed for training. Acoustic feature extraction and transcript tokenization happen dynamically during batching in the training notebook, so I'm not storing thousands of large feature tensors in memory -- keeps things scaling okay on Colab.

In [21]:
print("Training samples :", len(train_dataset))
print("Validation samples:", len(val_dataset))

print("\nFeatures:")
print(train_dataset.column_names)

print("\nNotebook 4 completed successfully.")

Training samples : 3638
Validation samples: 885

Features:
['audio_path', 'start', 'end', 'text']

Notebook 4 completed successfully.


In [ ]:
%whos

In [22]:
from datasets import DatasetDict

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset
})

SAVE_PATH = "/content/drive/MyDrive/legacyagent/datasets/legal_asr_dataset"

dataset.save_to_disk(SAVE_PATH)

print("Dataset saved to:")
print(SAVE_PATH)

Saving the dataset (0/1 shards):   0%|          | 0/3638 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/885 [00:00<?, ? examples/s]

Dataset saved to:
/content/drive/MyDrive/legacyagent/datasets/legal_asr_dataset
